# CausalRank — 从估计到排名：同时区间与排名置信集（Phase 3）

本 Notebook 演示 Phase 3 的排名推断内核：

1. **两两同时区间**：$\hat d_{ij}=\hat\mu_i-\hat\mu_j$，
   $\widehat{SE}(\hat d_{ij})=\sqrt{\hat\Sigma_{ii}+\hat\Sigma_{jj}-2\hat\Sigma_{ij}}$，
   Bonferroni 校正 $\hat d_{ij}\pm z_{1-\alpha/(2M)}\widehat{SE}$，$M=K(K-1)/2$；
2. **可确认优劣关系**：仅当同时区间整体位于 0 的一侧才宣称“确定优于/劣于”；
3. **保守排名置信区间**：$[L_i,U_i]=[1+B_i,\ K-W_i]$，
   $B_i$ 为可确认优于 $i$ 的路线数、$W_i$ 为可确认劣于 $i$ 的路线数；
4. **K=2 退化检验**：$M=1$ 时 Bonferroni 因子就是常规 $z_{1-\alpha/2}$；
5. **真并列红线**：`equal_effects` 场景下噪声不得产生确定排名。

> **红线**：不显著 ≠ 相等（`inconclusive` 不是相等的证据）；
> 点估计并列共享名次并显式标记，绝不按标签字母序伪造确定排序；
> bootstrap 中点排名第一的频次不是“最优概率”；单项 CI 重叠不能代替显著性检验。
> 排名结论依赖联合协方差与渐近正态近似，覆盖率以模拟检验为准（`scripts/run_rank_validation.py`）。

In [1]:
import numpy as np
import pandas as pd

from causalrank import CausalRank
from causalrank.inference import rank_from_estimates
from causalrank.simulation import make_dataset

pd.set_option("display.width", 130)

## 1. 完整分析：估计 + 联合 Bootstrap 协方差

排名推断消费的是 **Phase 2 的联合协方差** $\hat\Sigma$（K×K，标签规范序）。
关闭 `n_bootstrap` 或成功率过低时推断被阻断，`pairwise/ranks` 保持 `None`。

In [2]:
ds = make_dataset("confounded", n=2000, random_state=2026)
model = CausalRank(
    treatment="treatment",
    outcome="outcome",
    confounders=["x1", "x2", "x3", "x4"],
    cross_fit=3,
    n_bootstrap=100,
    confidence_level=0.95,
    higher_is_better=True,  # rank 1 = 最高转化率；成本型指标用 False
    random_state=2026,
)
result = model.fit(ds.df)
print("inference_available:", result.inference_available)
display(result.effects.to_dataframe().round(4))

inference_available: True


,label,n_treated,raw_mean,mu,se,ci_low,ci_high,weight_sum,ess
0,A,754,0.4549,0.4219,0.0176,0.3874,0.4563,2231.1808,584.4989
1,B,646,0.4582,0.5412,0.0303,0.4819,0.6006,2303.3496,252.5057
2,C,600,0.4350,0.3834,0.0206,0.3430,0.4238,2297.7461,394.6472


## 2. 两两同时区间（Bonferroni）

差值标准误使用**联合**协方差：$\widehat{SE}(\hat d_{ij})=\sqrt{\hat\Sigma_{ii}+\hat\Sigma_{jj}-2\hat\Sigma_{ij}}$，
臂间相关性被正确带入。`decision` 的语义是“更好”方向（受 `higher_is_better` 影响）：

- `CI_lower(i-j) > 0` → `i_better`（确认 $i$ 优于 $j$）；
- `CI_upper(i-j) < 0` → `j_better`；
- 其他 → `inconclusive`：**不能**解读为两者相等。

In [3]:
pairs = result.pairwise.to_dataframe()
display(pairs.round(4))
print("M =", result.pairwise.n_pairs, ", z_crit =", round(result.pairwise.z_crit, 4))
print("\n可确认关系（confirmed）：")
display(result.pairwise.confirmed[["label_i", "label_j", "diff", "ci_low", "ci_high", "decision"]].round(4))

,label_i,label_j,mu_i,mu_j,diff,se_diff,ci_low,ci_high,z_crit,n_pairs,confidence_level,decision,confirmed,point_tie,degenerate_se
0,A,B,0.4219,0.5412,-0.1194,0.0371,-0.2082,-0.0306,2.394,3,0.95,j_better,True,False,False
1,A,C,0.4219,0.3834,0.0384,0.0272,-0.0267,0.1036,2.394,3,0.95,inconclusive,False,False,False
2,B,C,0.5412,0.3834,0.1578,0.0376,0.0679,0.2478,2.394,3,0.95,i_better,True,False,False


M = 3 , z_crit = 2.394

可确认关系（confirmed）：


,label_i,label_j,diff,ci_low,ci_high,decision
0,A,B,-0.1194,-0.2082,-0.0306,j_better
1,B,C,0.1578,0.0679,0.2478,i_better


## 3. 保守排名置信区间 $[1+B_i,\ K-W_i]$

`point_rank` 为点排名（并列共享名次，`point_tie` 显式标记）；
`rank_low/rank_high` 是同时覆盖意义上的保守区间——分不出优劣时区间变宽，
这是诚实的答案，而不是按点估计强行给出全序。

In [4]:
ranks = result.ranks.to_dataframe()
display(ranks)
print(result.metadata["rank_note"])

,label,mu,point_rank,rank_low,rank_high,n_confirmed_better,n_confirmed_worse,point_tie
0,A,0.421865,2,2,3,1,0,False
1,B,0.541243,1,1,1,0,2,False
2,C,0.383418,3,2,3,1,0,False


Simultaneous Bonferroni pairwise intervals and conservative rank confidence sets [1 + B_i, K - W_i] computed from the joint bootstrap covariance (confidence_level=0.95). 'inconclusive' is not an equality claim; exact point-estimate ties share the same point rank and are flagged; the bootstrap frequency of point-rank 1 is not a probability of being best.


## 4. K=2 退化检验与手算对照

规格验收：K=2 时 $M=1$，$z_{1-\alpha/(2M)}=z_{1-\alpha/2}$，
同时区间退化为常规两方案比较。下面用手算数字验证内核。

In [5]:
from scipy.stats import norm

sigma = np.array([[0.0025, 0.0018], [0.0018, 0.0016]])
r2 = rank_from_estimates({"A": 0.5, "B": 0.4}, sigma)
display(r2.pairwise.to_dataframe().round(6))
display(r2.ranks.to_dataframe())
se_diff = np.sqrt(0.0025 + 0.0016 - 2 * 0.0018)
print("手算 se_diff =", se_diff)
print("手算 z_{0.975} =", norm.ppf(0.975), " 内核 z_crit =", r2.pairwise.z_crit)
assert abs(r2.pairwise.z_crit - norm.ppf(0.975)) < 1e-12

,label_i,label_j,mu_i,mu_j,diff,se_diff,ci_low,ci_high,z_crit,n_pairs,confidence_level,decision,confirmed,point_tie,degenerate_se
0,A,B,0.5,0.4,0.1,0.022361,0.056174,0.143826,1.959964,1,0.95,i_better,True,False,False


,label,mu,point_rank,rank_low,rank_high,n_confirmed_better,n_confirmed_worse,point_tie
0,A,0.5,1,1,1,0,1,False
1,B,0.4,2,2,2,1,0,False


手算 se_diff = 0.022360679774997907
手算 z_{0.975} = 1.959963984540054  内核 z_crit = 1.959963984540054


## 5. 真并列红线：`equal_effects` 场景

三个 Treatment 的潜在结果分布完全相同。此时真实名次不唯一（每个臂的
真实名次范围都是 $[1,K]$），正确输出是**没有可确认关系、排名区间保持全宽**。
若某次抽样里噪声“制造”出确定排名，会以 `confirmed` 对的形式暴露出来，
而不会被包装成结论。

In [6]:
ds_eq = make_dataset("equal_effects", n=1500, random_state=7)
result_eq = CausalRank(
    treatment="treatment",
    outcome="outcome",
    confounders=["x1", "x2", "x3", "x4"],
    cross_fit=3,
    n_bootstrap=80,
    bootstrap_min_successes=30,
    random_state=7,
).fit(ds_eq.df)
display(result_eq.ranks.to_dataframe())
print("confirmed pairs:", int(result_eq.pairwise.to_dataframe()["confirmed"].sum()))

,label,mu,point_rank,rank_low,rank_high,n_confirmed_better,n_confirmed_worse,point_tie
0,A,0.507972,2,1,3,0,0,False
1,B,0.444275,3,1,3,0,0,False
2,C,0.587871,1,1,3,0,0,False


confirmed pairs: 0


## 6. 边界与红线

1. `inconclusive` **不是**相等的证据；“差异不显著”与“效果相等”是两个命题；
2. 排名区间依赖**同时**差值区间与渐近正态近似，不是无条件保证；
   用 `scripts/run_rank_validation.py` 做覆盖率与假确认率的模拟检验；
3. 不要把 bootstrap 里点排名第一的频次当“最优概率”；
4. 单项 CI 重叠不能代替两两显著性检验；
5. 不变性（规格验收）：Treatment 重排只置换行；效果加共同常数不改变排名结论；
   `higher_is_better=False` 时优劣方向与名次同步翻转。

定义与协议见 `docs/methodology.md` 与 `docs/simulation_protocol.md`。